In [1]:
!pip install transformers

In [2]:
!pip install "transformers[torch]"

In [3]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [5]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-test.csv")

In [6]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [7]:
train_data.sample(5)

,id,dialogue,summary
10941,13680855,Sara: I wanna go to the ZOO\r\nHank: What the ...,Sara wants to go to the zoo. Hank will talk to...
14274,13681864,Alie: Where are my headphones?\r\nNat: I don't...,Alie cannot find her earphones and she believe...
3746,13818990,Harper: I'm cooking pad thai. You're welcome t...,Harper invites Sean and Joe to try the pad tha...
13397,13864655,Kelly: Mum's phone is off\nKelly: She told me ...,Kelly will buy dark chocolate for her mother.
8705,13682022,Jude: Did you say you’re going to take a vacat...,Ruth is going for a 1-week vacation to New Yor...


In [8]:
# random sampling

train_data= train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data= val_data.sample(n=500, random_state=42).reset_index(drop=True)


In [9]:
train_data.shape

(4000, 3)

# data preprocessing

In [10]:
import re

def clean_data(text):
  text = re.sub(r"\r\n", " ", text) #lines
  text = re.sub(r"\s+", " ", text) #spaces
  text = re.sub(r"<.*?>", " ", text) #html tags
  text = re.sub(r"\r\n", " ", text) #lines
  text = text.strip().lower()
  return text

In [11]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)


## tokenization

In [12]:
from transformers import AutoTokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [13]:
# raw data ==> tokenized input for fine tuning

def preprocess(data):
    inputs = tokenizer(
        data["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        data["summary"],
        padding="max_length",
        max_length=200,
        truncation=True
    )

    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": targets["input_ids"]
    }

In [14]:
train_dataset = train_data.apply(preprocess,axis=1).tolist()
val_dataset = val_data.apply(preprocess,axis=1).tolist()

In [15]:
train_dataset[1]

{'input_ids': [6234,
  10,
  78,
  405,
  1321,
  214,
  116,
  8,
  6093,
  19,
  352,
  12,
  1837,
  58,
  16585,
  10,
  12050,
  6,
  150,
  6,
  68,
  133,
  310,
  114,
  12,
  5,
  3,
  1050,
  2494,
  10,
  3,
  23,
  278,
  31,
  17,
  317,
  3,
  23,
  31,
  26,
  36,
  1638,
  16,
  48,
  5,
  6234,
  10,
  3,
  63,
  58,
  3,
  1050,
  2494,
  10,
  2492,
  66,
  8,
  1717,
  11,
  3224,
  3640,
  7,
  656,
  140,
  1227,
  19974,
  5,
  16585,
  10,
  78,
  25,
  31,
  60,
  78,
  7569,
  58,
  6234,
  10,
  3,
  75,
  31,
  2157,
  55,
  3,
  7,
  52,
  7,
  120,
  58,
  3,
  1050,
  2494,
  10,
  3,
  63,
  413,
  5,
  141,
  8,
  337,
  589,
  437,
  3,
  23,
  47,
  3,
  9,
  861,
  5,
  16585,
  10,
  2087,
  34,
  31,
  7,
  97,
  12,
  483,
  34,
  58,
  6234,
  10,
  17945,
  55,
  428,
  34,
  3,
  9,
  653,
  55,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
 

In [18]:
len(train_dataset[0]["input_ids"])

512

## model training

In [17]:
# we are dealing with the NLP task and also text generation task conditional generation


model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
from torch.distributed import is_available
import torch

if torch.cuda.is_available():
  print("GPU is available")
else:
  print("GPU is not available")

GPU is available


In [20]:
# training parameters & arguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay = 0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy= "epoch",
    save_strategy = "epoch",
    warmup_steps = 500

)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

## model training

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.533317,0.468975
2,0.300679,0.472918
3,0.282102,0.474998
4,0.273894,0.478272
5,0.268685,0.479241
6,0.265258,0.479263


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.8206558430989583, metrics={'train_runtime': 1426.3669, 'train_samples_per_second': 16.826, 'train_steps_per_second': 2.103, 'total_flos': 3248203235328000.0, 'train_loss': 0.8206558430989583, 'epoch': 6.0})

In [24]:
model.save_pretrained ("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

## testing the model

In [31]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) # clean

  #tokenize
  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  ).to(model.device)

  # Move input tensors to the same device as the model
  # if torch.cuda.is_available():
  #     inputs = {name: tensor.to(model.device) for name, tensor in inputs.items()}

  # generating the summary => token ids

  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 200,
      num_beams = 4,
      early_stopping = False
  )

  summary = tokenizer.decode(targets[0], skip_special_tokens = True)
  return summary

In [32]:
test_dialogue = """


**Alex:** Hey, have you noticed how many electric vehicles are on the road these days?

**Jordan:** Yeah, I've definitely seen more of them. I'm actually thinking about buying one.

**Alex:** Really? What's making you consider an EV?

**Jordan:** Mainly the lower running costs. Electricity is generally cheaper than gasoline, and I've heard maintenance costs are lower too.

**Alex:** That's true. Since EVs have fewer moving parts, you don't have to worry about things like oil changes.

**Jordan:** Exactly. But I'm a little concerned about charging. How long does it usually take?

**Alex:** It depends. If you charge at home overnight, it's usually ready by morning. Fast chargers can recharge the battery to around 80% in about 30 to 45 minutes, depending on the vehicle.

**Jordan:** That doesn't sound too bad. What about driving range?

**Alex:** Many modern EVs can travel between 300 and 500 kilometers on a single charge. It really depends on the model and driving conditions.

**Jordan:** I mostly drive to work and back, so I don't think range would be an issue.

**Alex:** Then an EV might be a good fit for you. Plus, they're much quieter and produce no tailpipe emissions.

**Jordan:** That's another reason I'm interested. I'd like to reduce my environmental impact.

**Alex:** Have you looked into government incentives? Some places offer tax credits or rebates for buying electric vehicles.

**Jordan:** I haven't yet, but I'll definitely check. That could make the purchase more affordable.

**Alex:** Also, make sure you have a convenient place to charge, especially if you live in an apartment.

**Jordan:** Good point. I'll ask my apartment management if they have charging stations or plans to install them.

**Alex:** Sounds like you've got a good plan. Let me know which model you choose.

**Jordan:** Will do! Maybe next time we meet, I'll be driving an electric car.
"""

summary = summarize_dialogue(test_dialogue)

print("summary: ", summary)

summary:  jordan has seen more electric vehicles on the road lately. he's considering buying an ev. he's concerned about the lower running costs.
